# Project 1 – Decision Trees and Random Forests

In [35]:
# Reload all modules without having to restart the kernel
%load_ext autoreload
%autoreload 2

# Imports
import numpy as np
import random
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix
from sklearn.model_selection import cross_val_score, train_test_split, StratifiedKFold

# My implementations
from decision_tree import DecisionTree
from random_forest import RandomForest

# Base seed
seed = 0
np.random.seed(seed)
random.seed(seed)

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## Dataset

Do data loading, exploration and preprocessing as you see fit.

Here is some code to load the dataset to get you started.

In [36]:
# Load dataset
data = np.genfromtxt("letters.csv", delimiter=",", dtype=float, names=True)

feature_names = list(data.dtype.names[:-1])
target_name = data.dtype.names[-1]

X = np.array([data[feature] for feature in feature_names]).T
y = data[target_name].astype(int)

print(f"Feature column names: {feature_names}")
print(f"Target column name: {target_name}")
print(f"X shape: {X.shape}")
print(f"y shape: {y.shape}")

Feature column names: ['xbox', 'ybox', 'width', 'high', 'onpix', 'xbar', 'ybar', 'x2bar', 'y2bar', 'xybar', 'x2ybr', 'xy2br', 'xege', 'xegvy', 'yege', 'yegvx']
Target column name: label
X shape: (2000, 16)
y shape: (2000,)


In [38]:
#DECISION TREES

# Train/validation split
#First split:
X_train, X_val_test, y_train, y_val_test = train_test_split(
    X, y, test_size=0.3, random_state=seed, shuffle=True, stratify=y) #train = 70%, VALIDATION = 30%
X_test, X_val, y_test, y_val = train_test_split(X_val_test, y_val_test, test_size=0.5, shuffle=True, stratify=y_val_test) #THIS SPLITS THE DATA INTO 15% TEST 15% VALIDATION
#15% validation 
#15% test


#shuffle is true: ensures the data is mixed before splitting → prevents ordered bias
#stratify = to the label class, makes the proportion of the labels equal and unbiased ###SUPER IMPORTANT
# this accounts for equal distribution of labels 1 or 0 across training test and validation 50 50.
#random_state=seed(able to be checked for assignment)

# StratifiedKFold ensures class proportions are preserved
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=seed) #********************SOURCE (WHY WE USE SKF)
# prepares 5-fold cross validation
# each fold keeps the same proportion in each class
# shuffle = true means that the seed ensures that it can be reproduced

# Decision Tree hyperparameters to test
dt_max_depths = [None, 4, 6, 8, 10, 12, 14]
#values that could be best for max depth and criteria
dt_criteria  = ["gini", "entropy"]

dt_hyper_parameters_combo = [
    {"max_depth": d, "criterion": c}
    for d in dt_max_depths for c in dt_criteria
]
#creating test+validation DATA to be constantly split up
X_train_val = np.concatenate([X_train, X_val], axis=0)
y_train_val = np.concatenate([y_train, y_val], axis=0)

all_scores = []
# stores cross validation results for each hyper parameter combination

for hyper_param in dt_hyper_parameters_combo:
    fold_accuracy = []
    fold_f1 = []
#this loop says that for each hyperpar it prepares lists to collect the results across folds 

    for train_index, val_index in cv.split(X_train_val, y_train_val):# this loop splits training data and validation data 5 times
        X_train_fold, X_val_fold = X_train_val[train_index], X_train_val[val_index]
        y_train_fold, y_val_fold = y_train_val[train_index], y_train_val[val_index]

        model = DecisionTree( #train DT with current hyperparameters defined in outer loop
            max_depth=hyper_param["max_depth"], 
            criterion=hyper_param["criterion"],
        )
        model.fit(X_train_fold, y_train_fold) #fits 
        y_predict = model.predict(X_val_fold) #predict

        fold_accuracy.append(accuracy_score(y_val_fold, y_predict)) #evaluate accuracy
        fold_f1.append(f1_score(y_val_fold, y_predict, average="macro", zero_division=0)) #evaluate f1 score ***********************@SOURCE, and zero div

    # Compute means for this hyperparam combo
    cv_acc = float(np.mean(fold_accuracy))
    cv_macroF1 = float(np.mean(fold_f1))

    all_scores.append({                 
        "cv_acc": cv_acc,                           #we have 16 of these. 2x8
        "cv_macroF1": cv_macroF1,
        "max_depth": hyper_param["max_depth"],
        "criterion": hyper_param["criterion"],
    })

# Pick best by accuracy, tie-break by macro-F1
best_score = all_scores[0] #we use all_scores[0] just so we have something to compare agenst
best_idx = 0
for i, row in enumerate(all_scores[1:], start=1):
    if (row["cv_acc"], row["cv_macroF1"]) > (best_score["cv_acc"], best_score["cv_macroF1"]):
        best_score = row
        best_idx = i

print("Best Decision Tree:",
      f"acc={best_score['cv_acc']:.4f}, macroF1={best_score['cv_macroF1']:.4f},",
      f"max_depth={best_score['max_depth']}, criterion={best_score['criterion']}")
    


Best Decision Tree: acc=0.9106, macroF1=0.9112, max_depth=12, criterion=entropy


*Random Forest*

In [39]:
#RANDOMFOREST hyperparameters to test
rf_n_estimators = [5,10,20] ##*****@SOURCE why do we use these values
rf_max_depts = [None,5,10]#*****@SOURCE why do we use these values
rf_criteria = ["gini","entropy"]
rf_max_features = ["sqrt","log2",None] #WHy do we use log2 and none

rf_hyper_parameters_combo = [
    {"n_estimators": n, "max_depth": d, "criterion": c, "max_features": mf}
    for n in rf_n_estimators
    for d in rf_max_depts
    for c in rf_criteria
    for mf in rf_max_features]

all_rf_scores = []
for hyper_param in rf_hyper_parameters_combo:
    fold_accuracy = []
    fold_f1 = []

    for train_index, val_index in cv.split(X_train_val, y_train_val):
        X_train_fold, X_val_fold = X_train_val[train_index], X_train_val[val_index]
        y_train_fold, y_val_fold = y_train_val[train_index], y_train_val[val_index]

        model = RandomForest(
            n_estimators=hyper_param["n_estimators"],
            max_depth=hyper_param["max_depth"], 
            criterion=hyper_param["criterion"],
            max_features=hyper_param["max_features"]
        )
        model.fit(X_train_fold, y_train_fold)
        y_predict = model.predict(X_val_fold)

        fold_accuracy.append(accuracy_score(y_val_fold, y_predict)) #evaluate accuracy
        fold_f1.append(f1_score(y_val_fold, y_predict, average="macro", zero_division=0))

    # Compute means for this hyperparam combo
    cv_acc = float(np.mean(fold_accuracy))
    cv_macroF1 = float(np.mean(fold_f1))

    all_rf_scores.append({
        "cv_acc": cv_acc,
        "cv_macroF1": cv_macroF1,
        "n_estimators": hyper_param["n_estimators"],
        "max_depth": hyper_param["max_depth"],
        "criterion": hyper_param["criterion"],
        "max_features": hyper_param["max_features"],
    })

# Pick best by accuracy, tie-break by macro-F1
best_rf_score = all_rf_scores[0]
for row in all_rf_scores[1:]:
    if (row["cv_acc"], row["cv_macroF1"]) > (best_rf_score["cv_acc"], best_rf_score["cv_macroF1"]):
        best_rf_score = row

print("Best Random Forest:",
      f"acc={best_rf_score['cv_acc']:.4f}, macroF1={best_rf_score['cv_macroF1']:.4f},",
      f"n_estimators={best_rf_score['n_estimators']},",
      f"max_depth={best_rf_score['max_depth']},",
      f"criterion={best_rf_score['criterion']},",
      f"max_features={best_rf_score['max_features']}")

Best Random Forest: acc=0.9629, macroF1=0.9632, n_estimators=20, max_depth=None, criterion=gini, max_features=sqrt


**in order for us to be able to compare our models to sklearn and for it to be fair we have to be able to have the same type of training for the model** Hence:

In [40]:
#SKLEARNS DECISION TREES

# StratifiedKFold ensures class proportions are preserved
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=seed) #********************SOURCE (WHY WE USE SKF)
# prepares 5-fold cross validation
# each fold keeps the same proportion in each class
# shuffle = true means that the seed ensures that it can be reproduced

# Decision Tree hyperparameters to test
dt_max_depths = [None, 4, 6, 8, 10, 12, 14]
#values that could be best for max depth and criteria
dt_criteria  = ["gini", "entropy"]

dt_hyper_parameters_combo = [
    {"max_depth": d, "criterion": c}
    for d in dt_max_depths for c in dt_criteria
]

all_scores_sklearn = []
# stores cross validation results for each hyper parameter combination

for hyper_param in dt_hyper_parameters_combo:
    fold_accuracy = []
    fold_f1 = []
#this loop says that for each hyperpar it prepares lists to collect the results across folds 

    for train_index, val_index in cv.split(X_train_val, y_train_val):# this loop splits training data and validation data 5 times
        X_train_fold, X_val_fold = X_train_val[train_index], X_train_val[val_index]
        y_train_fold, y_val_fold = y_train_val[train_index], y_train_val[val_index]

        model = DecisionTreeClassifier( #train DT with current hyperparameters defined in outer loop
            max_depth=hyper_param["max_depth"], 
            criterion=hyper_param["criterion"],
        )
        model.fit(X_train_fold, y_train_fold) #fits 
        y_predict = model.predict(X_val_fold) #predict

        fold_accuracy.append(accuracy_score(y_val_fold, y_predict)) #evaluate accuracy
        fold_f1.append(f1_score(y_val_fold, y_predict, average="macro", zero_division=0)) #evaluate f1 score ***********************@SOURCE, and zero div

    # Compute means for this hyperparam combo
    cv_acc = float(np.mean(fold_accuracy))
    cv_macroF1 = float(np.mean(fold_f1))

    all_scores_sklearn.append({                 
        "cv_acc": cv_acc,                           #we have 16 of these. 2x8
        "cv_macroF1": cv_macroF1,
        "max_depth": hyper_param["max_depth"],
        "criterion": hyper_param["criterion"],
    })

# Pick best by accuracy, tie-break by macro-F1
best_score_sklearn = all_scores_sklearn[0] #we use all_scores[0] just so we have something to compare agenst
best_idx = 0
for i, row in enumerate(all_scores_sklearn[1:], start=1):
    if (row["cv_acc"], row["cv_macroF1"]) > (best_score_sklearn["cv_acc"], best_score_sklearn["cv_macroF1"]):
        best_score_sklearn = row
        best_idx = i

print("Best SKLEARN Decision Tree:",
      f"acc={best_score_sklearn['cv_acc']:.4f}, macroF1={best_score_sklearn['cv_macroF1']:.4f},",
      f"max_depth={best_score_sklearn['max_depth']}, criterion={best_score_sklearn['criterion']}")
    

Best SKLEARN Decision Tree: acc=0.9171, macroF1=0.9178, max_depth=10, criterion=entropy


**MAKE THE SAME THING WITH SKLEARN RANDOM FOREST**

In [41]:
#SKLEARN RANDOMFOREST hyperparameters to test
rf_n_estimators = [5,10,20] ##*****@SOURCE why do we use these values
rf_max_depts = [None,5,10]#*****@SOURCE why do we use these values
rf_criteria = ["gini","entropy"]
rf_max_features = ["sqrt","log2",None] #WHy do we use log2 and none

rf_hyper_parameters_combo = [
    {"n_estimators": n, "max_depth": d, "criterion": c, "max_features": mf}
    for n in rf_n_estimators
    for d in rf_max_depts
    for c in rf_criteria
    for mf in rf_max_features]

all_rf_scores_sklearn = []
for hyper_param in rf_hyper_parameters_combo:
    fold_accuracy = []
    fold_f1 = []

    for train_index, val_index in cv.split(X_train_val, y_train_val):
        X_train_fold, X_val_fold = X_train_val[train_index], X_train_val[val_index]
        y_train_fold, y_val_fold = y_train_val[train_index], y_train_val[val_index]

        model = RandomForestClassifier(
            n_estimators=hyper_param["n_estimators"],
            max_depth=hyper_param["max_depth"], 
            criterion=hyper_param["criterion"],
            max_features=hyper_param["max_features"]
        )
        model.fit(X_train_fold, y_train_fold)
        y_predict = model.predict(X_val_fold)

        fold_accuracy.append(accuracy_score(y_val_fold, y_predict)) #evaluate accuracy
        fold_f1.append(f1_score(y_val_fold, y_predict, average="macro", zero_division=0))

    # Compute means for this hyperparam combo
    cv_acc = float(np.mean(fold_accuracy))
    cv_macroF1 = float(np.mean(fold_f1))

    all_rf_scores_sklearn.append({
        "cv_acc": cv_acc,
        "cv_macroF1": cv_macroF1,
        "n_estimators": hyper_param["n_estimators"],
        "max_depth": hyper_param["max_depth"],
        "criterion": hyper_param["criterion"],
        "max_features": hyper_param["max_features"],
    })

# Pick best by accuracy, tie-break by macro-F1
best_rf_score_sklearn = all_rf_scores_sklearn[0]
for row in all_rf_scores_sklearn[1:]:
    if (row["cv_acc"], row["cv_macroF1"]) > (best_rf_score_sklearn["cv_acc"], best_rf_score_sklearn["cv_macroF1"]):
        best_rf_score_sklearn = row

print("Best Sklearn Random Forest:",
      f"acc={best_rf_score_sklearn['cv_acc']:.4f}, macroF1={best_rf_score_sklearn['cv_macroF1']:.4f},",
      f"n_estimators={best_rf_score_sklearn['n_estimators']},",
      f"max_depth={best_rf_score_sklearn['max_depth']},",
      f"criterion={best_rf_score_sklearn['criterion']},",
      f"max_features={best_rf_score_sklearn['max_features']}")

Best Sklearn Random Forest: acc=0.9641, macroF1=0.9645, n_estimators=20, max_depth=None, criterion=entropy, max_features=log2


**now we can compare all 4 models because they have been trained in the same mannor**

In [42]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

#Getting out the test data from all the models, they all have the same test data due to the seed being the same

#3.2 Model Evaluation

#putting the training and validation data together
X_train_validationset = np.concatenate([X_train, X_val], axis =0) #used in hw1
y_train_validationset = np.concatenate([y_train, y_val], axis =0)# used in hw1

#_____#DT final implementation______#
dt_final_model = DecisionTree(max_depth=best_score["max_depth"],criterion=best_score["criterion"])
dt_final_model.fit(X_train_validationset,y_train_validationset)
dt_y_pred = dt_final_model.predict(X_test)

dt_final_accuracy = accuracy_score(y_test,dt_y_pred)
print("Our Decision Tree algorithm Accuracy:",dt_final_accuracy)

#_____#RF final implementation______#
rf_final_model = RandomForest(
    n_estimators = best_rf_score["n_estimators"],
    max_depth = best_rf_score["max_depth"],
    criterion = best_rf_score["criterion"],
    max_features = best_rf_score["max_features"]
)

rf_final_model.fit(X_train_validationset,y_train_validationset)
rf_y_pred = rf_final_model.predict(X_test)

rf_final_accuracy = accuracy_score(y_test,rf_y_pred)
print("Our Random Forest Tree alogirhtm Accuracy:",rf_final_accuracy)

#____SKlearn DT________#
sk_dt = DecisionTreeClassifier(max_depth = best_score_sklearn['max_depth'], criterion = best_score_sklearn["criterion"], random_state=seed)
sk_dt.fit(X_train_validationset, y_train_validationset)
sk_y_pred = sk_dt.predict(X_test)
print("For the Sklearn Decision Tree the Test Acc:", accuracy_score(y_test, sk_y_pred))

#____SKlearn RF________#
sk_rf = RandomForestClassifier(n_estimators = best_rf_score_sklearn['n_estimators'],
                                max_depth=best_rf_score_sklearn['max_depth'], 
                                criterion=best_rf_score_sklearn['criterion'], 
                                max_features=best_rf_score_sklearn['max_features'], 
                                random_state=seed)
sk_rf.fit(X_train_validationset, y_train_validationset)
sk_y_pred = sk_rf.predict(X_test)
print("Sklearn Random Forest Test Acc:", accuracy_score(y_test, sk_y_pred))


Our Decision Tree algorithm Accuracy: 0.92
Our Random Forest Tree alogirhtm Accuracy: 0.9633333333333334
For the Sklearn Decision Tree the Test Acc: 0.9233333333333333
Sklearn Random Forest Test Acc: 0.9633333333333334
